# Notebook de Simulação e Teste para o LiveTrader e o SetupScanner

Este notebook permite executar o fluxo de trabalho de dois componentes principais:
1. **LiveTrader (Passo a Passo)**: Para um único ativo escolhido, ideal para depuração detalhada.
2. **SetupScanner (Execução Completa)**: Para todos os ativos habilitados, ideal para obter uma visão geral das oportunidades atuais.

**Pré-requisitos:**
1. O terminal MetaTrader 5 deve estar aberto e logado.
2. Os modelos de produção devem ter sido gerados pelo script `train_model.py`.

In [ ]:
import sys
from pathlib import Path
import MetaTrader5 as mt5
from datetime import datetime
import logging

logging.basicConfig(level=logging.INFO)

# Adiciona a pasta 'src' ao path para permitir as importações dos nossos módulos
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.live_trader import LiveTrader
from src.setup_scanner import SetupScanner # <-- NOVA IMPORTAÇÃO

## Parte 1: Teste Assistido do LiveTrader (Ativo Único)
---
### Passo 1.1: Escolha o Ativo e Inicialize o Trader

**Ação:** Defina a variável `DATA_TICKER_PARA_TESTAR` com o ticker de DADOS HISTÓRICOS (ex: "WDO$").

In [ ]:
# --- ESCOLHA O ATIVO AQUI (use o ticker de dados históricos) ---
DATA_TICKER_PARA_TESTAR = "WDO$"
DATETIME_STR = "2025-10-20" # Data ('YYYY-MM-DD') ou Data e Hora ('YYYY-MM-DD HH:MM')
# ------------------------------------------------------------

trader = LiveTrader(config_path='configs/main.yaml')
is_initialized = trader.initialize()
asset_state_para_testar = None

if is_initialized and DATA_TICKER_PARA_TESTAR in trader.asset_states:
    print(f"Trader inicializado com sucesso. Foco do teste: {DATA_TICKER_PARA_TESTAR}")
    asset_state_para_testar = trader.asset_states[DATA_TICKER_PARA_TESTAR]
else:
    print(f"Falha ao inicializar ou ticker '{DATA_TICKER_PARA_TESTAR}' não encontrado/carregado.")

### Passo 1.2: Executar um Único Ciclo de Decisão (Single Tick)

In [ ]:
def run_single_tick(trader_instance, asset_state):
    if not asset_state:
        print("Estado do ativo não inicializado.")
        return

    data_ticker = asset_state['config']['ticker']
    live_config = asset_state['config']['live_trading']
    order_ticker = asset_state['config']['live_trading']['ticker_order']
    timeframe_str = live_config['timeframe_str']
    mt5_timeframe = trader_instance._get_mt5_timeframe_from_string(timeframe_str)
    
    print(f"--- Executando ciclo de decisão para: {data_ticker} ({timeframe_str}) ---")

    # Se o horário de execução for após 18:30, usar os dados do dia anterior(114), se não, utilizar 0
    # candle_position = 114 if datetime.now().time() > datetime.strptime("18:30", "%H:%M").time() else 0
    
    # Ajusta a posição do candle inicial de acordo com a datime_str informada
    candle_position = 0

    # 1. Buscar dados recentes
    print(f"Buscando candles de {data_ticker}...")
    latest_data = trader_instance.provider.get_rates(data_ticker, candle_position, 300, mt5_timeframe)

    if latest_data.empty: 
        print("Ticker de dados vazio, efetuando busca com o ticker de ordem.")
        latest_data = trader_instance.provider.get_rates(order_ticker, candle_position, 300, mt5_timeframe)
        if latest_data.empty:
            print("Ticker de ordem também vazio.")
            return    

    print(f"Último candle recebido: {latest_data.index[-1]}")
    display(latest_data.tail(3))

    last_candle_time = latest_data.index[-1].strftime('%Y-%m-%d %H:%M:%S')
    print(f"Último candle recebido: {last_candle_time}")
    display(latest_data.tail(3))

    # 2. Gerar features
    featured_data = asset_state["strategy"].define_features(latest_data)
    X_live = featured_data[asset_state["strategy"].get_feature_names()].dropna()
    if X_live.empty: 
        print("Dados insuficientes para gerar features.")
        return

    # 3. Gerar sinal
    print("\nGerando sinal com o modelo de IA...")
    signal = asset_state["model"].predict(X_live)[-1]
    signal_text = 'COMPRA' if signal == 1 else 'VENDA'

    # --- LÓGICA DE FALLBACK PARA O PREÇO SUGERIDO ---
    suggested_price = 0.0
    price_source = "N/A"
    symbol_info = mt5.symbol_info_tick(order_ticker)

    # 1ª Tentativa: Obter o preço do tick em tempo real
    if symbol_info and symbol_info.ask > 0 and symbol_info.bid > 0:
        suggested_price = symbol_info.ask if signal == 1 else symbol_info.bid
        price_source = "Tick (Tempo Real)"
    # 2ª Tentativa (Fallback): Usar o preço de fecho do último candle
    elif not latest_data.empty:
        suggested_price = latest_data['close'].iloc[-1]
        price_source = "Fechamento do Último Candle"

    stop_loss_pct = asset_state['config']['trading_rules']['stop_loss_pct']
    take_profit_pct = stop_loss_pct = asset_state['config']['trading_rules']['take_profit_pct']

    take_profit_price = suggested_price * (1 - take_profit_pct) if signal == 1 else suggested_price * take_profit_pct
    stop_price = suggested_price * (1 - stop_loss_pct) if signal == 1 else suggested_price * stop_loss_pct

    last_indicators = featured_data.iloc[-1]
    ma_values = {}
    ma_keys = ['ema_9', 'sma_20', 'sma_50', 'sma_200'] # Assumindo que a estratégia calcula estas MMs
    for key in ma_keys:
        if key in last_indicators:
            ma_values[key] = last_indicators[key]

    if ma_values:
        ma_str = " | ".join([f"{key.upper()}: {value:.2f}" for key, value in ma_values.items()])

    suggestion = []

    suggestion.append({
        "Tipo de Operação": signal_text,
        "Candle de Entrada": last_candle_time,        
        "Preço Sugerido de Entrada": f"{suggested_price:.2f}",
        "Preço de Stop": f"{stop_price:.2f}",
        "Preço de Saida": f"{take_profit_price}",        
        "Médias Móveis": ma_str,
        "Fonte do preço": price_source,
        "signal": signal, "timestamp": last_candle_time
    })
    
    print("SUGESTÃO DE ENTRADA")   
    print(f"Tipo de Operação: {signal_text}")
    print(f"Candle de Entrada : {last_candle_time}")        
    print(f"Preço Sugerido de Entrada: {suggested_price:.2f}")
    print(f"Preço de Stop: {stop_price}")
    print(f"Preço de Saida: {take_profit_price}")        
    if ma_values:
        ma_str = " | ".join([f"{key.upper()}: {value:.2f}" for key, value in ma_values.items()])
        print(f"Médias Móveis: {ma_str}")
    print(f"Fonte do preço: {price_source}")
    print(f"signal: {signal}, timestamp:{last_candle_time}")  
    

    # 4. Lógica de decisão
    #if asset_state["position"] is None:
    #    if signal == 1: trader_instance._execute_trade(data_ticker, 'BUY')
    #    elif signal == 0: trader_instance._execute_trade(data_ticker, 'SELL')
    #else:
    #    print(f"Posição já aberta para {data_ticker} ({asset_state['position']}).")

    print("--- Ciclo de decisão concluído ---")

    return suggestion[0]

### Passo 1.3: Executar o Teste de Ciclo Único

Execute esta célula para rodar a simulação do `LiveTrader` para o ativo selecionado.

In [ ]:
print("\n--- Iniciando Teste de Ciclo Único para o LiveTrader ---")
try:
    if is_initialized and asset_state_para_testar:
        suggestion = run_single_tick(trader, asset_state_para_testar)   
        print(suggestion)
except Exception as e:
    print(f"Erro durante o teste do LiveTrader: {e}")

In [ ]:
suggestion  # Imprime a sugestão gerada pelo LiveTrader

## Parte 3: Encerrar a Conexão
---

Ao final de todos os seus testes, execute esta célula para garantir que a conexão com o MetaTrader 5 seja encerrada corretamente.

In [ ]:
print("Encerrando conexão com o MetaTrader 5...")
mt5.shutdown()
print("Conexão encerrada.")

## Parte 2: Teste do Scanner de Setups (Todos os Ativos)
---
### Passo 2.1: Executar o Scanner

Esta célula irá instanciar o `SetupScanner` e executá-lo uma vez para **todos os ativos habilitados** no `main.yaml`, apresentando as sugestões que atenderem tanto ao sinal da IA quanto às regras de setup configuradas.

In [ ]:
"""
print("\n--- Iniciando Teste do Scanner de Setups ---")
try:
    # O scanner se conecta e desconecta do MT5 internamente.
    scanner = SetupScanner(config_path='configs/main.yaml')
    scanner.scan_all_assets()
except Exception as e:
    print(f"Erro durante a execução do scanner: {e}")
"""